# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [25]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [26]:

df['revenue'] = df['qty'] * df['price']

print(f'total revenue: ${df["revenue"].sum():,.2f}')
print(f'total units: {df["qty"].sum():,}')

total revenue: $8,520.00
total units: 783


### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [27]:

by_category = df.groupby('category').agg(
    revenue = ('revenue', 'sum'),
    units = ('qty', 'sum'),
)

by_category.sort_values('revenue', ascending=False, inplace=True)

by_category['share'] = by_category['revenue'] / by_category['revenue'].sum()
by_category


by_category.loc['total'] = by_category.sum()
by_category.loc['total', 'share'] = ''
by_category

print(by_category)

          revenue  units     share
category                          
Food       4293.0  362.0  0.503873
Merch      1771.5  158.0  0.207923
Drink      1554.0  178.0  0.182394
RainGear    901.5   85.0   0.10581
total      8520.0  783.0          


/tmp/ipykernel_1607/249308666.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  by_category.loc['total', 'share'] = ''


### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [28]:

by_vendor = df.groupby('vendor_id').agg(
    revenue = ('revenue', 'sum'),
    orders = ('qty', 'count'),
)
by_vendor['average'] = by_vendor['revenue'] / by_vendor['orders']
by_vendor

,revenue,orders,average
vendor_id,,,
V-01,2124.0,94,22.595745
V-05,1914.0,93,20.580645
V-10,2133.0,105,20.314286
V-18,2349.0,108,21.750000


### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [29]:
#print share of revenue from Merch as a percentage
print(f'The share of revenue from Merch is {by_category.loc["Merch", "share"] * 100:.1f}%')


The share of revenue from Merch is 20.8%


### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [30]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

joined = pd.merge(df, vendor_names, on='vendor_id', how='left', validate='many_to_one')
print(joined)

print(f'row count: {len(joined)}')
print(f'total revenue: ${joined["revenue"].sum():,.2f}')


    vendor_id  category  qty  price  revenue      vendor_name
0        V-10     Drink    2   24.0     48.0  Cav Merch North
1        V-18  RainGear    1   12.0     12.0              NaN
2        V-18     Drink    3    4.5     13.5              NaN
3        V-10      Food    2   12.0     24.0  Cav Merch North
4        V-18     Drink    3    7.5     22.5              NaN
..        ...       ...  ...    ...      ...              ...
395      V-18     Merch    1   12.0     12.0              NaN
396      V-01     Merch    2   24.0     48.0     Hoos Burgers
397      V-10      Food    3    7.5     22.5  Cav Merch North
398      V-18     Merch    2   24.0     48.0              NaN
399      V-10     Drink    1    7.5      7.5  Cav Merch North

[400 rows x 6 columns]
row count: 400
total revenue: $8,520.00


**The unmatched vendor, and what I did about it:** The unmatched vendor is V-18, I would find the name of this vendor and update that in vendor_names.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [31]:
pivot = pd.pivot_table(
    joined,
    values='revenue',
    index='vendor_id',
    columns='category',
    aggfunc='sum',
    margins=True,
    margins_name='Total',
)
pivot

category,Drink,Food,Merch,RainGear,Total
vendor_id,,,,,
V-01,171.0,1338.0,373.5,241.5,2124.0
V-05,298.5,882.0,489.0,244.5,1914.0
V-10,502.5,1054.5,400.5,175.5,2133.0
V-18,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [32]:
len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category.drop('total')['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.


**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.


Focus more on food sales which is where most of the revenue is coming from, given it represents 50% of the total revenue. In addition to focusing on food sales, vendors should observe V-18 to see how they are gaining higher sales even though they're average order value is 21 compared to 20 for other vendors.

Q2 is the least trustworthy because it does not tell us anything about the sales of the specific vendors, but rather the general categories, which varies among each vendor.